In [1]:
import sys
sys.executable

'C:\\Users\\munsl\\venvs\\credit-ml\\Scripts\\python.exe'

In [2]:
import pandas as pd

df_raw = pd.read_excel("../data/default of credit card clients.xls", header=None)
df_raw.head(3)

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,NaN,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X15,X16,X17,X18,X19,X20,X21,X22,X23,Y
1,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
2,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1


This data is raw, it has the column names on row 1 but they could be in row 0. Row 0 currentyl has generic labels. I will fix this by choosing a header value.

In [3]:
df = pd.read_excel("../data/default of credit card clients.xls", header=1)
df.head(3)

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0


I chose header=1 because row 0 had all the generic X1,X2,X3... labels, and row 1 clearly had all the real names I need.

In [4]:
df.shape

(30000, 25)

I expected 23 features but got 25. This is due to the ID column being a row counter and the default payment next month column is the thing I'm trying to predict.

## Data Audit

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_AMT3        

Pandas calls each column int64, meaning whole numbers, but e.g. per the UCI variable descriptions, for Sex, 1 = male and 2 = female, Education 1 = Grad school 2 = Uni 3 = high school 4 = others, marriage is 1 = married 2 = single 3 = others, these are all categorical data, i.e EDUCATION 4 is not twice as much as EDUCATION 2. They are different pieces of info. We can also see that each column has 30000 non null entries, meaning there are no blank cells. However, this does not guarantee no bad values, which are codes that aren't in the UCI document.

In [7]:
df.isna().sum().sum()

np.int64(0)

I checked there were no missing values by using a method to count missing cells across all columns, called isna(), and the result outputted 0, verifying the earlier result from .info(), there were no missing values in my dataset.

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df.drop(columns="ID").duplicated().sum()

np.int64(35)

In [10]:
df[df.drop(columns="ID").duplicated(keep=False)]

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
18,19,360000,2,1,1,49,1,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,0
627,628,230000,1,1,1,39,-1,-1,-1,-1,...,660,660,660,660,660,660,660,660,660,0
839,840,500000,1,1,1,43,1,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,1
1093,1094,360000,1,2,1,41,1,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,1
1211,1212,160000,2,3,2,26,-1,-1,-1,-1,...,390,390,390,390,390,390,390,390,390,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28779,28780,200000,2,1,1,36,1,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,0
28983,28984,80000,2,3,1,42,-2,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,0
29265,29266,180000,1,2,1,26,-1,-1,-1,-1,...,396,396,396,396,396,396,396,396,396,0
29823,29824,220000,1,1,1,42,1,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,1


Markdown: I checked for duplicates using the .duplicated().sum() method, returning 0. I thought this was suspicious for a large dataset of 30000 entries, so I thought about what could be hiding them. ID is different for each row, so that would hide duplicates, so I dropped that column in the search, which resulted in showing me 35 duplicate rows. I then called the table of these duplicates, showing 70 rows because its both copies of the duplicates(35x2). In the table, a lot of the entries for BILL_AMT and PAY_AMT are 0. This leads me to believe a lot of these duplicates are just inactive customers, with identical variables like age, sex, marriage, education. There are also some duplicates with the same small bill and payment amount every month. These probably aren't data entry errors but I will still drop them before splitting to avoid data leakage. 35/30000 is only around 0.12% of the data so it won't impact things much.

In [12]:
df["default payment next month"].value_counts()

default payment next month
0    23364
1     6636
Name: count, dtype: int64

In [13]:
df["default payment next month"].value_counts(normalize=True)

default payment next month
0    0.7788
1    0.2212
Name: proportion, dtype: float64

Above, I calculated the proportion of customers who defaulted vs those who didn't. We can see 22.12% of customers here did default, which confirms the ~22% I stated in my proposal. Also, looking at the split of 78% and 22%, if we took a lazy model that only predicted "no default" for each customer, it's accuracy would be 78%, while catching 0 defaulters. This indicates accuracy alone is a misleading metric for having a strong prediction model. Therefore, I will use recall, precision F1 and PR-AUC along with a cost metric instead. I'll also use a stratified split so the 78/22 ratio remains consistent across training, validation and test data.

## Checking Category Codes

In [15]:
df["EDUCATION"].value_counts().sort_index()

EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64

From education, I can see there are some undocumented codes: 0, 5, 6. Being 345 rows, they make up 1.15% of the data, which is small, so in preprocessing I'll merge them into EDUCATION = 4 "others", because "others" is a catch all for these unknown values, and having a separate category this small, the model wouldn't have much to learn from so it makes sense to include them in "others".

In [16]:
df["MARRIAGE"].value_counts().sort_index()

MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64

Now in marriage, we have an undocumented code 0. It's 54 rows, making up only 0.18% of the data. Like education, I will merge them later into MARRIAGE = 3 "others", because again "others" is a catch all.

In [17]:
df["PAY_0"].value_counts().sort_index()

PAY_0
-2     2759
-1     5686
 0    14737
 1     3688
 2     2667
 3      322
 4       76
 5       26
 6       11
 7        9
 8       19
Name: count, dtype: int64

From PAY_0, we can see all the codes. There are undocumented codes, for -2 and 0, as UCI only defines -1 and 1-9. These make up 58% of rows which is too big to merge or drop, they must mean something. I'm not sure exactly what it could be, I would suggest it could be an inactive customer for -2 based on the duplicate rows earlier where PAY_ columns were mostly -2 and the bills were all 0. For PAY_0 = 0, I'm not yet sure but will do checks below.

In [20]:
df[df["PAY_0"]==-2]["BILL_AMT1"].describe()

count      2759.000000
mean       8793.052555
std       28224.565390
min      -15308.000000
25%           0.000000
50%        1179.000000
75%        5819.000000
max      478030.000000
Name: BILL_AMT1, dtype: float64

In [21]:
df["BILL_AMT1"].describe()

count     30000.000000
mean      51223.330900
std       73635.860576
min     -165580.000000
25%        3558.750000
50%       22381.500000
75%       67091.000000
max      964511.000000
Name: BILL_AMT1, dtype: float64

For PAY_0 = -2, I checked BILL_AMT1 for these customers compared to all customers. What I found, was the median for -2 is 1179, compared to 22381 for all customers, telling us -2 customers have much smaller bills. Only about 25% are 0 or lower, so they aren't completely inactive like my original theory stated, low activity is a better description. The negative bills above are also just money owed by the bank to a customer, or overpaid customers, suggesting a credit balance. It suggests it's not a data error.

In [22]:
df[df["PAY_0"]==0]["BILL_AMT1"].describe()

count     14737.000000
mean      78417.889462
std       82054.920294
min         260.000000
25%       22233.000000
50%       49605.000000
75%      104691.000000
max      964511.000000
Name: BILL_AMT1, dtype: float64

This time, I compared PAY_0 = 0 against all customers. The median here is 49605, much higher than all customers median of 22381, almost double. It's also a strong contrast to the -2 customers(1179). PAY_0 = 0 also has a minimum of 260, so each customer here has a bill, suggesting to me these are active customers carrying a balance, not paying late, just keeping up to date with payments without clearing the full balance. This is consistent with revolving credit.

My final assumed order of the codes is -2 being low activity, -1 being paid duly, 0 being active, carrying a balance, and 1-9 being in months late. Seeing as the order seems sensible here, being most to least risky, I'm going to keep the undocumented PAY codes as they are rather than merging or dropping. They clearly have meaningful info within the data. However, I'd like to state this is an assumption of mine, as UCI doesn't document -2 and 0, so this is just based on my bill amount checks above.